In [7]:
# ============================================================
# PromptSentinel — Notebook 4
# Chunked Embedding Experiment
#
# Author: Leesha Mogha
# Institution: IMS Ghaziabad (University Course Campus)
# Project: PromptSentinel (v4)
# ============================================================
#
# What this notebook does:
# Tests whether semantic embeddings outperform TF-IDF on the
# short, creative human-written jailbreaks where Notebook 3
# showed vocabulary mismatch is the bottleneck.
#
# Research question answered here:
# RQ2 extension: Does meaning-based detection close the gap
# that TF-IDF leaves on short human-written jailbreaks?
#
# Secondary question:
# Does chunked reading of long prompts improve detection
# compared to encoding the full prompt as one unit?

In [2]:
# Section 1: Setup

!pip install sentence-transformers datasets pandas numpy scikit-learn -q

from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    recall_score,
    precision_score,
    f1_score
)
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# all-MiniLM-L6-v2: fast, 384-dim, runs comfortably on T4 free tier
# Chosen deliberately over larger models — fair comparison with TF-IDF
# requires keeping compute similar. Larger models tested in future work.
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
embedder   = SentenceTransformer(MODEL_NAME)

print(f"Embedder loaded: {MODEL_NAME}")
print(f"Embedding dim  : {embedder.get_sentence_embedding_dimension()}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder loaded: sentence-transformers/all-MiniLM-L6-v2
Embedding dim  : 384


In [3]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
embedder = SentenceTransformer(MODEL_NAME)
print(f"Embedder re-initialized: {MODEL_NAME}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedder re-initialized: sentence-transformers/all-MiniLM-L6-v2


In [4]:
# Section 2: Load Data
# Same split as Notebook 3 — training set excludes TrustAIRLab entirely.
# TrustAIRLab is the held-out human-written test set throughout.

from google.colab import files
from pathlib import Path
import pandas as pd # Added this line to import pandas

print("Upload compressed_data.csv.gz (from Notebook 1)")
uploaded   = files.upload()
df_all     = pd.read_csv('compressed_data.csv.gz')

# Remove TrustAIRLab from training — same as Notebook 3
df_train   = df_all[df_all['source'] != 'trustairlab'].reset_index(drop=True)
df_train   = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Training set : {len(df_train):,} prompts")
print(df_train['label'].value_counts())
print()

# Reload TrustAIRLab as clean held-out test set
jailbreak = load_dataset(
    'TrustAIRLab/in-the-wild-jailbreak-prompts',
    'jailbreak_2023_05_07', split='train'
)
regular = load_dataset(
    'TrustAIRLab/in-the-wild-jailbreak-prompts',
    'regular_2023_05_07', split='train'
)

df_unsafe      = jailbreak.to_pandas()[['prompt']]
df_safe        = regular.to_pandas()[['prompt']]
df_unsafe['label'] = 'unsafe'
df_safe['label']   = 'safe'

df_test = pd.concat([df_unsafe, df_safe], ignore_index=True)
df_test = df_test.dropna()
df_test = df_test[df_test['prompt'].str.strip() != ''].reset_index(drop=True)

print(f"Test set (TrustAIRLab) : {len(df_test):,} prompts")
print(df_test['label'].value_counts())

Upload compressed_data.csv.gz (from Notebook 1)


Saving compressed_data.csv.gz to compressed_data.csv.gz
Training set : 110,060 prompts
label
safe      57653
unsafe    52407
Name: count, dtype: int64



README.md:   0%|          | 0.00/9.54k [00:00<?, ?B/s]

jailbreak_2023_05_07/train-00000-of-0000(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/666 [00:00<?, ? examples/s]

regular_2023_05_07/train-00000-of-00001.(…):   0%|          | 0.00/3.26M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5721 [00:00<?, ? examples/s]

Test set (TrustAIRLab) : 6,387 prompts
label
safe      5721
unsafe     666
Name: count, dtype: int64


In [5]:
# Section 3: Embed Training Data and Train Classifier
#
# Embedding 110k prompts takes ~15-20 min on T4.
# batch_size=256 keeps GPU utilisation high without OOM.
# show_progress_bar gives a live ETA so you know it's running.

print("Embedding training prompts — this takes ~15 min on T4...")
print()

X_train_emb = embedder.encode(
    df_train['prompt'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)
y_train = df_train['label'].values

print()
print(f"Training embedding matrix : {X_train_emb.shape}")

# Logistic Regression — same hyperparameters as Notebooks 2 and 3
# so the comparison is purely about the feature representation
model_emb = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0)
model_emb.fit(X_train_emb, y_train)

print("Embedding classifier trained.")

Embedding training prompts — this takes ~15 min on T4...



Batches:   0%|          | 0/430 [00:00<?, ?it/s]


Training embedding matrix : (110060, 384)
Embedding classifier trained.


In [6]:
# Section 4: Test on TrustAIRLab — Full Prompt Encoding
#
# Baseline embedding result: encode each prompt as one unit.
# This is the direct comparison with Notebook 3's TF-IDF result.

print("Embedding test prompts...")

X_test_emb = embedder.encode(
    df_test['prompt'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)
y_test = df_test['label'].values

y_pred_emb = model_emb.predict(X_test_emb)

print()
print("=== Embedding classifier — full prompt ===")
print()
print(classification_report(y_test, y_pred_emb, zero_division=0))

recall_emb_full    = recall_score(y_test, y_pred_emb, pos_label='unsafe', zero_division=0)
precision_emb_full = precision_score(y_test, y_pred_emb, pos_label='unsafe', zero_division=0)
f1_emb_full        = f1_score(y_test, y_pred_emb, pos_label='unsafe', zero_division=0)

Embedding test prompts...


Batches:   0%|          | 0/25 [00:00<?, ?it/s]


=== Embedding classifier — full prompt ===

              precision    recall  f1-score   support

        safe       0.90      0.42      0.58      5721
      unsafe       0.11      0.58      0.18       666

    accuracy                           0.44      6387
   macro avg       0.50      0.50      0.38      6387
weighted avg       0.81      0.44      0.54      6387



In [8]:
# Section 5: TF-IDF vs Embeddings — Short Prompt Breakdown
#
# This is the core comparison for RQ2.
# Notebook 3 showed recall was 0.568 on 0-50 word prompts with TF-IDF.
# We test whether embeddings close that gap.

from sklearn.feature_extraction.text import TfidfVectorizer

# Rebuild TF-IDF baseline (same settings as Notebook 3)
print("Training TF-IDF baseline for comparison...")

vectorizer_tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
model_tfidf      = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0)

model_tfidf.fit(
    vectorizer_tfidf.fit_transform(df_train['prompt']),
    y_train
)
y_pred_tfidf = model_tfidf.predict(
    vectorizer_tfidf.transform(df_test['prompt'])
)

# Add predictions and length to test dataframe
df_test = df_test.copy()
df_test['pred_tfidf'] = y_pred_tfidf
df_test['pred_emb']   = y_pred_emb
df_test['length']     = df_test['prompt'].str.split().str.len()

# Focus on unsafe prompts — misses are what matter
df_unsafe_test = df_test[df_test['label'] == 'unsafe'].copy()

bins   = [0, 50, 100, 200, 400, 800, 9999]
labels = ['0–50', '51–100', '101–200', '201–400', '401–800', '800+']
df_unsafe_test['length_bin'] = pd.cut(
    df_unsafe_test['length'], bins=bins, labels=labels
)

def recall_by_bin(pred_col):
    return (
        df_unsafe_test
        .groupby('length_bin', observed=True)
        .apply(lambda g: (g[pred_col] == 'unsafe').sum() / len(g))
    )

recall_tfidf = recall_by_bin('pred_tfidf')
recall_emb   = recall_by_bin('pred_emb')
counts       = df_unsafe_test['length_bin'].value_counts().sort_index()

print()
print("=== Recall by prompt length — TF-IDF vs Embeddings ===")
print()
print(f"{'Bin':<12} {'TF-IDF':>8} {'Embed':>8} {'Delta':>8} {'n':>6}")
print("-" * 48)

for bin_label in labels:
    r_tf  = recall_tfidf.get(bin_label, float('nan'))
    r_emb = recall_emb.get(bin_label, float('nan'))
    delta = r_emb - r_tf if not (np.isnan(r_tf) or np.isnan(r_emb)) else float('nan')
    n     = counts.get(bin_label, 0)
    delta_str = f"{delta:+.3f}" if not np.isnan(delta) else "  n/a"
    print(f"{bin_label:<12} {r_tf:>8.3f} {r_emb:>8.3f} {delta_str:>8} {n:>6}")

print()
# Highlight the short-prompt bin specifically
short_tfidf = recall_tfidf.get('0–50', float('nan'))
short_emb   = recall_emb.get('0–50', float('nan'))
if not (np.isnan(short_tfidf) or np.isnan(short_emb)):
    gap = short_emb - short_tfidf
    print(f"Short prompt gap (0–50 words): TF-IDF {short_tfidf:.3f} → Embedding {short_emb:.3f}  ({gap:+.3f})")

Training TF-IDF baseline for comparison...

=== Recall by prompt length — TF-IDF vs Embeddings ===

Bin            TF-IDF    Embed    Delta      n
------------------------------------------------
0–50            0.568    0.500   -0.068     44
51–100          0.487    0.618   +0.132     76
101–200         0.580    0.587   +0.007    150
201–400         0.722    0.594   -0.128    187
401–800         0.724    0.595   -0.129    163
800+            0.717    0.457   -0.261     46

Short prompt gap (0–50 words): TF-IDF 0.568 → Embedding 0.500  (-0.068)


In [9]:
# Section 6: Chunked Encoding Experiment
#
# Hypothesis: encoding a long prompt as one unit loses detail
# from the early/late parts of the text. Chunking into overlapping
# windows and max-pooling the unsafe probability may recover signal
# that full-prompt encoding misses.
#
# Design:
#   - Split each prompt into overlapping word windows
#   - Embed each chunk independently
#   - Predict unsafe probability for each chunk
#   - Final prediction = max unsafe probability across all chunks
#   - Compare to full-prompt encoding on the same test set

CHUNK_SIZE    = 80   # words per chunk
CHUNK_OVERLAP = 20   # words of overlap between consecutive chunks

def chunk_prompt(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split a prompt into overlapping word windows."""
    words  = text.split()
    if len(words) <= size:
        return [text]
    step   = size - overlap
    chunks = []
    for i in range(0, len(words) - overlap, step):
        chunk = ' '.join(words[i : i + size])
        if chunk.strip():
            chunks.append(chunk)
    return chunks

def predict_chunked(prompts, embedder, model, batch_size=256):
    """
    For each prompt:
      1. Split into chunks
      2. Embed all chunks in one batched call (efficient)
      3. Predict unsafe probability for each chunk
      4. Return max unsafe probability → final label
    """
    all_chunks   = []
    prompt_index = []   # which prompt each chunk belongs to

    for i, prompt in enumerate(prompts):
        chunks = chunk_prompt(prompt)
        all_chunks.extend(chunks)
        prompt_index.extend([i] * len(chunks))

    # Embed everything in one pass
    all_embeddings = embedder.encode(
        all_chunks,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    # Get unsafe probability for each chunk
    unsafe_idx   = list(model.classes_).index('unsafe')
    chunk_probs  = model.predict_proba(all_embeddings)[:, unsafe_idx]

    # Max-pool across chunks for each prompt
    n_prompts    = len(prompts)
    max_probs    = np.zeros(n_prompts)
    prompt_index = np.array(prompt_index)

    for i in range(n_prompts):
        mask = prompt_index == i
        if mask.any():
            max_probs[i] = chunk_probs[mask].max()

    # Threshold at 0.5
    labels = np.where(max_probs >= 0.5, 'unsafe', 'safe')
    return labels, max_probs

print("Running chunked prediction on TrustAIRLab test set...")
print(f"  Chunk size    : {CHUNK_SIZE} words")
print(f"  Chunk overlap : {CHUNK_OVERLAP} words")
print()

y_pred_chunked, chunk_probs = predict_chunked(
    df_test['prompt'].tolist(),
    embedder,
    model_emb
)

print()
print("=== Embedding classifier — chunked ===")
print()
print(classification_report(y_test, y_pred_chunked, zero_division=0))

recall_chunked    = recall_score(y_test, y_pred_chunked, pos_label='unsafe', zero_division=0)
precision_chunked = precision_score(y_test, y_pred_chunked, pos_label='unsafe', zero_division=0)
f1_chunked        = f1_score(y_test, y_pred_chunked, pos_label='unsafe', zero_division=0)

# Does chunking help specifically on long prompts?
df_test['pred_chunked'] = y_pred_chunked
df_unsafe_test2 = df_test[df_test['label'] == 'unsafe'].copy()
df_unsafe_test2['length_bin'] = pd.cut(
    df_unsafe_test2['length'], bins=bins, labels=labels
)

recall_chunked_by_len = (
    df_unsafe_test2
    .groupby('length_bin', observed=True)
    .apply(lambda g: (g['pred_chunked'] == 'unsafe').sum() / len(g))
)

print()
print("=== Full vs chunked recall on long prompts (400+ words) ===")
print()
for bin_label in ['401–800', '800+']:
    r_full    = recall_emb.get(bin_label, float('nan'))
    r_chunked = recall_chunked_by_len.get(bin_label, float('nan'))
    if not (np.isnan(r_full) or np.isnan(r_chunked)):
        print(f"  {bin_label} words:  full {r_full:.3f}  →  chunked {r_chunked:.3f}  ({r_chunked - r_full:+.3f})")

Running chunked prediction on TrustAIRLab test set...
  Chunk size    : 80 words
  Chunk overlap : 20 words



Batches:   0%|          | 0/84 [00:00<?, ?it/s]


=== Embedding classifier — chunked ===

              precision    recall  f1-score   support

        safe       0.94      0.28      0.43      5721
      unsafe       0.12      0.85      0.21       666

    accuracy                           0.34      6387
   macro avg       0.53      0.57      0.32      6387
weighted avg       0.86      0.34      0.41      6387


=== Full vs chunked recall on long prompts (400+ words) ===

  401–800 words:  full 0.595  →  chunked 0.975  (+0.380)
  800+ words:  full 0.457  →  chunked 1.000  (+0.543)


In [11]:
# Sanity check: how does chunking affect precision specifically?

print("=== Precision check: chunked vs full, by length bin ===")
print()

df_safe_test = df_test[df_test['label'] == 'safe'].copy()
df_safe_test['length_bin'] = pd.cut(df_safe_test['length'], bins=bins, labels=labels)

for bin_label in ['401–800', '800+']:
    safe_subset = df_safe_test[df_safe_test['length_bin'] == bin_label]
    if len(safe_subset) == 0:
        continue
    false_positive_rate_full    = (safe_subset['pred_emb']     == 'unsafe').mean()
    false_positive_rate_chunked = (safe_subset['pred_chunked'] == 'unsafe').mean()
    print(f"  {bin_label} words (n={len(safe_subset)}):")
    print(f"    Full prompt    false positive rate: {false_positive_rate_full:.3f}")
    print(f"    Chunked        false positive rate: {false_positive_rate_chunked:.3f}")
    print()

=== Precision check: chunked vs full, by length bin ===

  401–800 words (n=406):
    Full prompt    false positive rate: 0.443
    Chunked        false positive rate: 0.901

  800+ words (n=101):
    Full prompt    false positive rate: 0.327
    Chunked        false positive rate: 0.970



In [12]:
# Section 7 (revised): Final Comparison and RQ2 Answer

print("=" * 60)
print("NOTEBOOK 4 SUMMARY (revised)")
print("=" * 60)
print()

results = pd.DataFrame([
    {
        'Method'    : 'TF-IDF (Notebook 3 baseline)',
        'Recall'    : round(recall_score(y_test, y_pred_tfidf,   pos_label='unsafe', zero_division=0), 3),
        'Precision' : round(precision_score(y_test, y_pred_tfidf, pos_label='unsafe', zero_division=0), 3),
        'F1'        : round(f1_score(y_test, y_pred_tfidf,       pos_label='unsafe', zero_division=0), 3),
    },
    {
        'Method'    : 'Embedding — full prompt',
        'Recall'    : round(recall_emb_full, 3),
        'Precision' : round(precision_emb_full, 3),
        'F1'        : round(f1_emb_full, 3),
    },
    {
        'Method'    : 'Embedding — chunked (max-pool)',
        'Recall'    : round(recall_chunked, 3),
        'Precision' : round(precision_chunked, 3),
        'F1'        : round(f1_chunked, 3),
    },
])

print(results.to_string(index=False))
print()

print("=" * 60)
print("FINDING 1: Precision collapses out-of-distribution")
print("=" * 60)
print()
print("All three v4-based methods show precision of 0.10-0.17 on")
print("TrustAIRLab, despite v2's behaviour and Notebook 2's near-perfect")
print("results on WildJailbreak-derived test data.")
print()
print("Cause: v4 training is dominated by WildJailbreak (100k of 110k")
print("prompts). Its 'safe' examples are synthetic vanilla/adversarial")
print("benign prompts, which look nothing like TrustAIRLab's real safe")
print("prompts (everyday ChatGPT usage from Reddit/Discord — coding")
print("questions, roleplay, creative writing).")
print()
print("The model has learned 'safe = WildJailbreak-style benign', not")
print("'safe' in general. This is a distribution mismatch in the SAFE")
print("class, not the unsafe class.")
print()

print("=" * 60)
print("FINDING 2: Embeddings do not close the short-prompt gap")
print("=" * 60)
print()
print(f"Short-prompt (0-50 words) recall: TF-IDF {short_tfidf:.3f} vs Embedding {short_emb_val:.3f}")
print(f"Gap: {short_emb_val - short_tfidf:+.3f}")
print()
print("Switching feature representation does not fix the short-prompt")
print("problem. This supports a training-distribution explanation over")
print("a vocabulary explanation: the model has not seen the SEMANTIC")
print("patterns of short human-written jailbreaks, regardless of how")
print("the text is encoded.")
print()

print("=" * 60)
print("FINDING 3: Chunking trades recall for precision catastrophically")
print("=" * 60)
print()
print(f"{'Length bin':<12} {'FP rate (full)':>16} {'FP rate (chunked)':>18}")
print("-" * 48)
print(f"{'401-800':<12} {0.443:>16.3f} {0.901:>18.3f}")
print(f"{'800+':<12} {0.327:>16.3f} {0.970:>18.3f}")
print()
print("Max-pooling across chunks means a long prompt is flagged unsafe")
print("if ANY chunk scores above threshold. With baseline precision")
print("already low (~0.10-0.12), this compounds: on 800+ word prompts,")
print("97% of genuinely SAFE prompts are now misclassified as unsafe.")
print()
print("The recall gain (+0.380) reported in the initial run is an")
print("artifact of the classifier becoming a near-constant 'unsafe'")
print("predictor for long text — not a genuine detection improvement.")
print()
print("Chunking with max-pooling is NOT a viable strategy without a")
print("much higher per-chunk decision threshold or an aggregation")
print("method other than max (e.g. mean, or majority vote).")
print()

print("=" * 60)
print("RQ2 ANSWER")
print("=" * 60)
print()
print("Does performance drop on longer human-written jailbreaks?")
print("The framing of RQ2 needs revision. The dominant effect is not")
print("prompt length — it is a distributional mismatch in what the")
print("model has learned to recognise as 'safe'. This affects prompts")
print("of all lengths, but is most visible in long prompts (where")
print("vocabulary diversity makes it easiest for the model to find")
print("SOME phrase that resembles training-set 'unsafe' vocabulary).")
print()
print("This reframes the threat model mismatch argument: v4's data")
print("quality improvement (Notebook 1) increased volume and reduced")
print("class imbalance, but did not increase DIVERSITY of the safe")
print("class in a way that generalises to real user behaviour.")
print()

print("Next: Notebook 5 — LLM-as-judge for borderline cases (RQ3)")
print("Given Finding 1, the LLM judge's main value may be reducing")
print("false positives on safe prompts that superficially resemble")
print("WildJailbreak's unsafe vocabulary — not catching missed unsafe")
print("prompts.")

NOTEBOOK 4 SUMMARY (revised)

                        Method  Recall  Precision    F1
  TF-IDF (Notebook 3 baseline)   0.653      0.174 0.275
       Embedding — full prompt   0.580      0.105 0.178
Embedding — chunked (max-pool)   0.848      0.121 0.212

FINDING 1: Precision collapses out-of-distribution

All three v4-based methods show precision of 0.10-0.17 on
TrustAIRLab, despite v2's behaviour and Notebook 2's near-perfect
results on WildJailbreak-derived test data.

Cause: v4 training is dominated by WildJailbreak (100k of 110k
prompts). Its 'safe' examples are synthetic vanilla/adversarial
benign prompts, which look nothing like TrustAIRLab's real safe
prompts (everyday ChatGPT usage from Reddit/Discord — coding
questions, roleplay, creative writing).

The model has learned 'safe = WildJailbreak-style benign', not
'safe' in general. This is a distribution mismatch in the SAFE
class, not the unsafe class.

FINDING 2: Embeddings do not close the short-prompt gap

Short-prompt (0-50